In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import pickle
import warnings
import json
warnings.filterwarnings("ignore")

In [2]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import load_model

2026-09-22 19:48:54.654442: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-22 19:48:55.128741: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-22 19:48:57.580319: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
df = pd.read_csv("/mnt/c/Office 2019/DS_PW/DL_Practice/Fake_Email_Detector/dataset/clean_data/spam_email.csv")
df

,body,label
0,"Buck up, your troubles caused by small dimensi...",Spam
1,\nUpgrade your sex and pleasures with these te...,Spam
2,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,Spam
3,Would anyone object to removing .so from this ...,Safe
4,\nWelcomeFastShippingCustomerSupport\nhttp://7...,Spam
...,...,...
63371,date a lonely housewife always wanted to date ...,Safe
63372,request submitted : access request for anita ....,Safe
63373,"re : important - prc mtg hi dorn & john , as y...",Safe
63374,press clippings - letter on californian utilit...,Safe


In [4]:
df.body[0]

'Buck up, your troubles caused by small dimension will soon be over!\nBecome a lover no woman will be able to resist!\nhttp://whitedone.com/\n\n\ncome. Even as Nazi tanks were rolling down the streets, the dreamersphilosopher or a journalist. He was still not sure.I do the same.'

In [5]:
df = df[df.body.notnull()]
df = df[df.body !="empty"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    df['body'], df['label'], test_size=0.001, random_state=42)

In [7]:
X_train.shape,X_test.shape

((62764,), (63,))

In [8]:
X_train = X_train.str.lower()
X_test = X_test.str.lower()

In [9]:
def remove__number(txt):
    if pd.isna(txt):
        return ""

    txt = str(txt)
    txt = re.sub(r'\d+', '', txt)
    txt = re.sub(r'["“”\'‘’]', '', txt)

    return txt

X_train = X_train.apply(remove__number)
X_test = X_test.apply(remove__number)


In [10]:
translator = str.maketrans('','',string.punctuation)
X_train = X_train.apply(lambda x: x.translate(translator))
X_test = X_test.apply(lambda x: x.translate(translator))

In [ ]:
voc_size = 298424
tokenizer = Tokenizer(num_words= voc_size)
tokenizer.fit_on_texts(X_train)

In [12]:
X_train_squence = tokenizer.texts_to_sequences(X_train)
X_test_squence = tokenizer.texts_to_sequences(X_test)

In [13]:
import json

word_index = tokenizer.word_index

with open("word_index.json", "w", encoding="utf-8") as f:
    json.dump(word_index, f, ensure_ascii=False)

In [14]:
max_len = 500

X_train = pad_sequences(
    X_train_squence,
    maxlen=max_len,
    padding="post",
    truncating="post"
)
X_test = pad_sequences(
    X_test_squence,
    maxlen=max_len,
    padding="post",
    truncating="post"
)

In [15]:
y_train = y_train.apply(lambda x: 1 if x == 'Spam' else 0)
y_test = y_test.apply(lambda x: 1 if x == 'Spam' else 0)

In [16]:
y_train = to_categorical(y_train,num_classes=2)
y_test = to_categorical(y_test,num_classes=2)

In [17]:
em_dim = 50
rnn_unit = 128


In [18]:
model = Sequential([
    Embedding(voc_size,em_dim,input_length=max_len),
    LSTM(units=rnn_unit),
    Dense(units=2,activation='softmax')
])

I0000 00:00:1790106559.508684    1819 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1763 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6


In [19]:
model.compile(
    optimizer= 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [20]:
checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor="accuracy",
    mode="max",
    save_best_only=True,
    verbose=1
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=1,
    batch_size=8,
    callbacks=[checkpoint]
)

In [ ]:
with open("training_history.json", "w") as f:
    json.dump(history.history, f)

In [23]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.save(filepath='final_model.h5')

In [ ]:
pred = model.predict(X_test,verbose=0)

2026-09-22 19:49:21.250954: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 92600


array([[0.49672654, 0.5032735 ],
       [0.49672654, 0.5032735 ],
       [0.4967265 , 0.5032735 ],
       [0.49672654, 0.5032735 ],
       [0.50057584, 0.4994242 ],
       [0.49672654, 0.5032735 ],
       [0.49672654, 0.5032735 ],
       [0.49672654, 0.5032735 ],
       [0.49672654, 0.5032735 ],
       [0.49672648, 0.50327355],
       [0.50160915, 0.49839088],
       [0.49672648, 0.50327355],
       [0.49672633, 0.50327367],
       [0.49672654, 0.5032735 ],
       [0.49672648, 0.50327355],
       [0.49672675, 0.50327325],
       [0.49672657, 0.5032734 ],
       [0.49672654, 0.5032735 ],
       [0.49672604, 0.50327396],
       [0.49501935, 0.5049807 ],
       [0.49672654, 0.5032735 ],
       [0.49672654, 0.5032735 ],
       [0.49672654, 0.5032735 ],
       [0.49672654, 0.5032735 ],
       [0.49672654, 0.5032735 ],
       [0.49672657, 0.5032734 ],
       [0.49672654, 0.5032735 ],
       [0.49672657, 0.5032734 ],
       [0.49672657, 0.5032734 ],
       [0.49921897, 0.500781  ],
       [0.

In [ ]:
pred1 = (pred > 0.5).astype(int)

array([[0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [1, 0],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [1, 0],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [1, 0],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [1, 0],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [1, 0],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1]])

In [27]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,pred1)

0.3492063492063492

In [28]:
import tensorflow as tf
import tf2onnx

max_len = 500
model = tf.keras.models.load_model("model1.h5")

input_signature = [
    tf.TensorSpec(
        shape=[None, max_len],
        dtype=tf.int32,
        name="input"
    )
]

tf2onnx.convert.from_keras(
    model,
    input_signature=input_signature,
    opset=17,
    output_path="best_model.onnx"
)

print("ONNX model saved!")

KeyError: 'keras_tensor_9'

In [ ]:
print(fddfdfdfdf)

In [ ]:
import tensorflow as tf
import tf2onnx

# model ko ek baar build/run karao
_ = model(tf.zeros((1, max_len), dtype=tf.int32))

# New Functional model
inputs = tf.keras.Input(
    shape=(max_len,),
    dtype=tf.int32,
    name="input"
)

outputs = model(inputs)

onnx_model = tf2onnx.convert.from_keras(
    tf.keras.Model(inputs, outputs),
    input_signature=[
        tf.TensorSpec(
            [None, max_len],
            tf.int32,
            name="input"
        )
    ],
    opset=17,
    output_path="best_model.onnx"
)

print("ONNX conversion successful!")

2026-09-22 19:14:06.544871: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 92600
I0000 00:00:1790104446.883018   10010 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
I0000 00:00:1790104446.883550   10010 single_machine.cc:376] Starting new session
I0000 00:00:1790104446.884601   10010 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1763 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1790104449.379961   10010 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1763 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1790104450.103547   10010 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
I0000 00:00:1790104450.103933   10010 single_machine.cc:376] Starting new sessio

ONNX conversion successful!


In [ ]:
model.export("saved_model")

INFO:tensorflow:Assets written to: saved_model/assets


INFO:tensorflow:Assets written to: saved_model/assets


Saved artifact at 'saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 500), dtype=tf.float32, name='input_layer_4')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  130523204370896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130523204372432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130523204372240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130523204374544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130523204372048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130523204373200: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [29]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import tensorflow as tf

print("GPU:", tf.config.list_physical_devices("GPU"))

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [30]:
import tensorflow as tf

with tf.device("/CPU:0"):
    model = tf.keras.models.load_model(
        "model1.h5"
    )

print(model.input_shape)
print(model.output_shape)

(None, 500)
(None, 2)


In [31]:
import numpy as np

dummy = np.zeros(
    (1, max_len),
    dtype=np.int32
)

with tf.device("/CPU:0"):
    output = model(dummy)

print(output.shape)

(1, 2)


In [35]:
with tf.device("/CPU:0"):

    cpu_model = tf.keras.Sequential([
        
        tf.keras.layers.Input(
            shape=(max_len,),
            dtype=tf.int32
        ),

        tf.keras.layers.Embedding(
            input_dim=voc_size,
            output_dim=em_dim
        ),

        tf.keras.layers.LSTM(
            units=rnn_unit
        ),

        tf.keras.layers.Dense(
            units=2,
            activation="softmax"
        )
    ])

In [36]:
cpu_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 500, 50)        │    14,921,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 128)            │        91,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,013,106 (57.27 MB)

 Trainable params: 15,013,106 (57.27 MB)

 Non-trainable params: 0 (0.00 B)

In [37]:
cpu_model.set_weights(
    model.get_weights()
)

In [38]:
print(
    np.max(
        np.abs(
            model(dummy).numpy()
            -
            cpu_model(dummy).numpy()
        )
    )
)

0.0


In [40]:
inputs = tf.keras.Input(
    shape=(max_len,),
    dtype=tf.int32,
    name="input"
)

outputs = cpu_model(inputs)

onnx_model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="fake_email_detector"
)

In [43]:
print("Input name:", onnx_model.inputs[0].name)
print("Output name:", onnx_model.outputs[0].name)

Input name: input
Output name: keras_tensor_24


In [45]:
import tensorflow as tf

inputs = tf.keras.Input(
    shape=(max_len,),
    dtype=tf.int32,
    name="input"
)

x = cpu_model(inputs)

outputs = tf.keras.layers.Activation(
    "linear",
    name="output"
)(x)

onnx_model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="fake_email_detector"
)

In [46]:
print("Input:", onnx_model.inputs[0].name)
print("Output:", onnx_model.outputs[0].name)

Input: input
Output: keras_tensor_26


In [47]:
import tf2onnx
import tensorflow as tf

input_signature = [
    tf.TensorSpec(
        shape=(None, max_len),
        dtype=tf.int32,
        name="input"
    )
]

spec = (tf.TensorSpec(
    (None, max_len),
    tf.int32,
    name="input"
),)

@tf.function(input_signature=spec)
def model_fn(x):
    return {"output": onnx_model(x)}

In [48]:
concrete_func = model_fn.get_concrete_function()

In [50]:
@tf.function(input_signature=spec)
def model_fn(x):
    return {"output": onnx_model(x)}

In [52]:
tf2onnx.convert.from_function(
    model_fn,
    input_signature=spec,
    opset=17,
    output_path="models/best_model.onnx"
)

I0000 00:00:1790107219.349800    1819 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
I0000 00:00:1790107219.356138    1819 single_machine.cc:376] Starting new session
I0000 00:00:1790107219.371793    1819 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1763 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1790107221.032767    1819 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1763 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1790107221.338566    1819 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
I0000 00:00:1790107221.338770    1819 single_machine.cc:376] Starting new session
I0000 00:00:1790107221.339427    1819 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:

(ModelProto(ir_version=8, opset_import={'': 17, 'ai.onnx.ml': 2}, producer_name='tf2onnx', producer_version='1.17.0 None', graph=GraphProto('tf2onnx', input=<1 inputs>, output=<1 outputs>, initializer=<10 initializers>, node=<24 nodes>)),
 None)

In [4]:
import onnx

m = onnx.load("models/best_model.onnx")

print("✅ ONNX loaded")
print("Inputs:", [x.name for x in m.graph.input])
print("Outputs:", [x.name for x in m.graph.output])
print("Ops:", sorted(set(node.op_type for node in m.graph.node)))

✅ ONNX loaded
Inputs: ['input']
Outputs: ['output']
Ops: ['Add', 'Cast', 'Concat', 'CudnnRNNV3', 'Expand', 'Gather', 'Less', 'MatMul', 'Mul', 'Not', 'Shape', 'Slice', 'Softmax', 'Squeeze', 'Unsqueeze']


In [5]:
cpu_model = m
for layer in cpu_model.layers:
    print(
        layer.name,
        "->",
        type(layer).__name__,
        getattr(layer, "units", None),
        getattr(layer, "return_sequences", None)
    )

AttributeError: layers

In [ ]:
m